[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/quadric_collision/quadric_collision.ipynb)

# Collision of Two Ellipsoids

This example decides whether two planar ellipsoids stand apart, touch, or overlap from the family of blends joining their dual quadrics. Each ellipsoid is modelled through its tangent planes, and linearly blending the first dual ellipsoid into the second sweeps through a one-parameter family of conics sharing every common tangent of the pair. While the two ellipsoids stand apart, an intermediate blend forms a separating hyperbola between them; once they overlap, every intermediate blend remains an ellipse bridging the overlap; and at grazing contact, the intermediate blend degenerates at the shared tangent and locates the contact point.

Points have type `Antivector-` and planes have type `Plane+`, so a dual quadric is a duality map `Point- <- Plane+` that vanishes under pairing on every tangent plane (`plane & dual(plane) == 0`) and sends each tangent plane to its contact point `dual(plane)`. Rigid motions transport the map as `motor >> dual(motor << Plane)`. Along the linear blend `first * (1 - share) + second * share`, the determinant of `blend.dual()` is a cubic polynomial in `share` fixed by four evaluations, whose concave peak on `(0, 1)` is positive when apart, zero at contact, and negative when overlapping (Wang, Wang, and Kim, 2001).

In homogeneous $3 \times 3$ matrix notation, each placed dual ellipse is assembled as $C_i^* = M_i \operatorname{diag}(a_i^2, b_i^2, -1) M_i^{\mathsf{T}}$ from an explicit rigid transformation matrix $M_i = \begin{bmatrix} R(\theta_i) & \mathbf{t}_i \\ \mathbf{0}^{\mathsf{T}} & 1 \end{bmatrix}$, tangency reads $\mathbf{p}^{\mathsf{T}} C_i^* \mathbf{p} = 0$ on homogeneous line-coordinate column vectors $\mathbf{p} = [n_x, n_y, -d]^{\mathsf{T}}$, and separation is tested from the characteristic polynomial $\det((1 - \lambda) C_1^* + \lambda C_2^*)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from numga import NumpyContext
from numga.algebras import PGA2D as ga

from examples.quadrics.quadric_collision import render, scenarios
from examples.util.animation import show_animation

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Plane = ga.gatype.vector()                                                     # Vector+
Point = ga.gatype.antivector()                                                 # Antivector-
Motor = ga.gatype.rotor()                                                      # Rotor+
# The dual quadric maps a tangent plane to its contact point; the primal polarity maps a point
# to its polar plane.
Quadric = ga.gatype((Point, Plane))                                            # Point- <- Plane+
Polarity = ga.gatype((Plane, Point))                                           # Plane+ <- Point-
# The plane at infinity, the unit-weight origin, and the coordinate directions along x and y.
infinity = mv.w                                                                # [] Plane+
origin = mv.w.dual()                                                           # [] Point-
direction_x, direction_y = mv.direction([1, 0]), mv.direction([0, 1])          # [] Point-, Point-

## Dual ellipsoids and outward tangent planes

An origin-centred dual ellipsoid combines the two coordinate direction dyads weighted by the squared semi-axes, minus the origin dyad, and a rigid motor transports the entire duality map into place. Among all planes sharing a given unit normal, two satisfy `plane & dual(plane) == 0` on opposite sides of the centre `dual(infinity)`; selecting the root further along the normal picks the outward tangent plane regardless of any nonzero scale factor on `dual`.

In [ ]:
demo_semi, demo_shift, demo_turn = (2.0, 1.2), (1.5, -0.8), np.radians(35)
facing_angles = np.radians([0, 45, 90, 135, 210, 315])
n_normals = len(facing_angles)

In [ ]:
def rest_ellipsoid(semi_x: float, semi_y: float) -> Quadric:
    """An origin-centred dual ellipsoid (`[] Point- <- Plane+`) combines the two axis direction dyads weighted by their squared semi-axes, minus the origin dyad."""
    return direction_x * (Plane & direction_x) * semi_x**2 + direction_y * (Plane & direction_y) * semi_y**2 - origin * (Plane & origin)


def placement_motor(shift_x: float, shift_y: float, angle: float) -> Motor:
    """A rigid motion (`[] Motor+`) turns about the origin by `angle` and translates by `(shift_x, shift_y)`."""
    translator = ((infinity ^ (mv.x * shift_x + mv.y * shift_y)) * -0.5).exp()   # [] Motor+
    rotor = (mv.xy * (-angle / 2)).exp()                                         # [] Motor+
    return translator * rotor


demo_motor = placement_motor(*demo_shift, demo_turn)                           # [] Motor+
demo_upright = rest_ellipsoid(*demo_semi)                                      # [] Point- <- Plane+
demo_dual = demo_motor >> demo_upright(demo_motor << Plane)                    # [] Point- <- Plane+

In [ ]:
def tangent_plane(dual: Quadric, normal: Plane) -> Plane:
    """The outward tangent plane (`[...] Plane+`) of `dual` along `normal` is invariant under rescaling or negating `dual`.

    Planes sharing `normal`'s direction have the form `unit - infinity * offset`. Tangency is quadratic in `offset`, `a * offset**2 - 2 * b * offset + c == 0`, whose coefficients pair `infinity` and `unit` across `dual`; the two roots straddle the centre offset `b / a`, and the outward root lies beyond it along `unit`.
    """
    unit = normal.normalized()                                                 # [...] Plane+
    a, b, c = infinity & dual(infinity), unit & dual(infinity), unit & dual(unit)   # [...] Scalar+, Scalar+, Scalar+
    return unit - infinity * (b / a + ((b * b - a * c) / (a * a)).square_root())


facing_rotors = (mv.xy * (-facing_angles / 2)).exp()                           # [n_normals] Motor+
demo_normals = facing_rotors >> mv.x                                           # [n_normals] Plane+
demo_tangents = tangent_plane(demo_dual, demo_normals)                         # [n_normals] Plane+
print("support offsets along the six normals:", -(demo_tangents & origin).to_array())

## Classifying poses from the cubic blend determinant

Translating a turned second ellipsoid so that its contact point with `-normal` coincides with the first ellipsoid's contact point with `normal` places the pair in exact tangency; shifting along `normal` then produces apart, touching, and overlapping poses. Evaluating `quadric_blend.dual().det()` at four shares recovers the cubic polynomial exactly, and its stationary peak on `(0, 1)` classifies each pose by sign.

In [ ]:
first_semi = scenarios.first_semi
first_centre = scenarios.first_centre
first_turn = scenarios.first_turn
second_semi = scenarios.second_semi
second_turn = scenarios.second_turn
contact_angle = scenarios.contact_angle
pose_offsets = scenarios.pose_offsets
n_poses = scenarios.n_poses
n_frames = scenarios.n_frames
slide_offsets = scenarios.slide_offsets
frame_duration_ms = scenarios.frame_duration_ms
cubic_samples = scenarios.cubic_samples
n_shares = scenarios.n_shares

In [ ]:
def quadric_blend(first: Quadric, second: Quadric, share: Scalar) -> Quadric:
    """Linearly interpolating `first` and `second` by `share` forms their intermediate dual quadric (`[...] Point- <- Plane+`)."""
    return first * (1 - share) + second * share


def cubic_peak(values: Scalar) -> tuple[Scalar, Scalar]:
    """Four determinant samples at shares `0, 1, 2, -1` along axis `0` (`[4, ...] Scalar+`) fix the concave cubic between `0` and `1` and locate its peak share and value (`[...] Scalar+, Scalar+`).

    Writing the stationary root of `3 * c3 * share**2 + 2 * c2 * share + c1` as `c1 / ((c2 * c2 - 3 * c3 * c1).square_root() - c2)` keeps the peak finite when the cubic term `c3` vanishes.
    """
    y0, y1, y2, y3 = values                                                    # [...] Scalar+, Scalar+, Scalar+, Scalar+
    c3 = (3 * y0 - 3 * y1 + y2 - y3) / 6                                       # [...] Scalar+
    c2 = -y0 + (y1 + y3) / 2                                                   # [...] Scalar+
    c1 = -y0 / 2 + y1 - y2 / 6 - y3 / 3                                        # [...] Scalar+
    c0 = y0                                                                    # [...] Scalar+
    peak = c1 / ((c2 * c2 - 3 * c3 * c1).square_root() - c2)                   # [...] Scalar+
    return peak, ((c3 * peak + c2) * peak + c1) * peak + c0


placement = placement_motor(*first_centre, first_turn)                         # [] Motor+
first_shape = rest_ellipsoid(*first_semi)                                      # [] Point- <- Plane+
first_ellipsoid = placement >> first_shape(placement << Plane)                 # [] Point- <- Plane+
contact_rotor = (mv.xy * (-contact_angle / 2)).exp()                           # [] Motor+
normal = contact_rotor >> mv.x                                                 # [] Plane+
second_ellipsoid = rest_ellipsoid(*second_semi)                                # [] Point- <- Plane+
turn = placement_motor(0.0, 0.0, second_turn)                                  # [] Motor+
turned_second = turn >> second_ellipsoid(turn << Plane)                        # [] Point- <- Plane+
# Translate the turned second ellipsoid so its opposite-normal contact lands on the first's contact.
target = first_ellipsoid(tangent_plane(first_ellipsoid, normal))               # [] Point-
start = turned_second(tangent_plane(turned_second, -normal))                   # [] Point-
displacement = target.normalized() - start.normalized()                        # [] Point-
translator = ((infinity ^ displacement.dual()) * -0.5).exp()                   # [] Motor+
touch_motor = translator * turn                                                # [] Motor+
# Shift along `normal` into apart, touching, and overlapping poses.
shifts = ((infinity ^ (normal * pose_offsets)) * -0.5).exp()                   # [n_poses] Motor+
poses = shifts * touch_motor                                                   # [n_poses] Motor+
placed_second = poses >> second_ellipsoid(poses << Plane)                      # [n_poses] Point- <- Plane+
sample_shares = mv.scalar(cubic_samples)[:, None]                              # [4, 1] Scalar+
peak_shares, peak_dets = cubic_peak(quadric_blend(first_ellipsoid, placed_second, sample_shares).dual().det())   # [n_poses] Scalar+, Scalar+
print("peak share per pose:", peak_shares.to_array())
print("peak determinant (apart, touching, overlapping):", peak_dets.to_array())

## Sliding contact and intermediate blend conics

At the touching pose the peak blend is singular: its trailing singular plane has zero image under the blend, making it the shared tangent plane, and both ellipsoids map that plane to the same contact point. Sweeping the offset smoothly from apart to overlapping streams the moving ellipsoid alongside the cubic determinant curve.

In [ ]:
_, _, planes = quadric_blend(first_ellipsoid, placed_second[1], peak_shares[1]).dual().svd()
contact_plane = planes[-1].normalized()                                        # [] Plane+
contact_point = first_ellipsoid(contact_plane).normalized()                    # [] Point-
first_tangent = tangent_plane(first_ellipsoid, normal)                         # [] Plane+
shares = mv.scalar(np.linspace(0.001, 0.999, n_shares)[:, None])               # [n_shares] Scalar+
frame_samples = mv.scalar(cubic_samples)                                       # [4] Scalar+
recorded_states = []


def slide_states(offsets: np.ndarray) -> Iterator[tuple[Quadric, Plane, Plane, Polarity, Scalar, Scalar, Scalar]]:
    """Yields the placed second ellipsoid, its facing tangent, the midplane, the peak primal polarity, and the determinant curve and peak at each offset."""
    for offset in offsets:
        shift = ((infinity ^ (normal * offset)) * -0.5).exp()                  # [] Motor+
        pose = shift * touch_motor                                             # [] Motor+
        moving_second = pose >> second_ellipsoid(pose << Plane)                # [] Point- <- Plane+
        peak_share, peak_det = cubic_peak(quadric_blend(first_ellipsoid, moving_second, frame_samples).dual().det())   # [] Scalar+, Scalar+
        second_tangent = tangent_plane(moving_second, -normal)                 # [] Plane+
        midplane = ((first_tangent - second_tangent) * 0.5).normalized()       # [] Plane+
        yield (
            moving_second,
            second_tangent,
            midplane,
            quadric_blend(first_ellipsoid, moving_second, peak_share).inverse(),
            quadric_blend(first_ellipsoid, moving_second, shares).dual().det(),
            peak_share,
            peak_det,
        )


def streamed_states(states: Iterator) -> Iterator:
    """Yields each state from `states` while recording it in `recorded_states` for downstream checks."""
    for state in states:
        recorded_states.append(state)
        yield state


show_animation(
    render.collision_animation(first_ellipsoid, first_tangent, shares, streamed_states(slide_states(slide_offsets))),
    "quadric_collision",
    frame_duration_ms,
    n_frames=n_frames,
    forward_then_back=True,
);

While the blue and orange ellipsoids stand apart, the peak blend appears in green as a separating hyperbola between them, accompanied by the solid green midplane halfway between their dashed facing tangents and a positive peak on the right-hand determinant curve. As the orange ellipsoid closes the dotted gap along the normal, the hyperbola narrows onto the shared tangent and the cubic peak touches zero; once the ellipsoids overlap, the peak drops below zero and the peak blend turns into a purple ellipse bridging their intersection.

In [ ]:
# checks
# Every outward tangent contains its contact point, the rescaled dual produces identical tangents,
# the cubic peak determinant classifies apart/touching/overlapping poses, and the shared tangent
# touches both ellipsoids at a single contact point.
assert ((demo_tangents & demo_dual(demo_tangents)).abs() < 1e-10).all()
assert ((tangent_plane(demo_dual * -2.5, demo_normals) - demo_tangents).norm() < 1e-12).all()
assert peak_dets[0] > 0 and peak_dets[2] < 0
assert peak_dets[1].abs() < 1e-12
assert (contact_plane & first_ellipsoid(contact_plane)).abs() < 1e-12
assert (contact_plane & placed_second[1](contact_plane)).abs() < 1e-12
assert (placed_second[1](contact_plane).normalized() & contact_point).norm() < 1e-12
assert len(recorded_states) == n_frames and recorded_states[0][6] > 0 and recorded_states[-1][6] < 0